# Phân tích khách hàng: Cohort & RFM (biểu đồ)

Notebook đọc 2 file CSV trong `../data/`, làm sạch **đúng quy tắc của `sql/02_clean.sql`**, tính lại cohort và RFM bằng pandas để **đối chiếu với kết quả SQL**, và lưu 4 biểu đồ vào `../images/`.

Chạy: **Restart → Run All**.

## 0. Thư viện và cấu hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({"font.family": "DejaVu Sans", "axes.spines.top": False, "axes.spines.right": False})
DATA, IMG = Path("../data"), Path("../images")
IMG.mkdir(exist_ok=True)
BLUE, ORANGE, GREY = "#2F6CA8", "#C46A1A", "#9AA5B1"

## 1. Đọc và làm sạch dữ liệu
Kết quả phải khớp SQL: 9.903 đơn, 4.403 khách, 4.639.399.000 VND.

In [ ]:
# ---------- 1. Đọc và làm sạch: đúng quy tắc của sql/02_clean.sql ----------
orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
customers = pd.read_csv(DATA / "customers.csv")

clean = (orders[(orders["net_value"] > 0) & ~orders["order_id"].str.startswith("TEST")]
         .drop_duplicates(subset="order_id", keep="first"))
clean = clean[clean["status"] == "Hoàn thành"].copy()
clean["month_idx"] = clean["order_date"].dt.year * 12 + clean["order_date"].dt.month
clean["ym"] = clean["order_date"].dt.strftime("%Y-%m")
print("Đơn sạch:", len(clean), "| Khách:", clean["customer_id"].nunique(),
      "| Doanh thu thuần (VND):", f"{clean['net_value'].sum():,}")   # 9.903 | 4.403 | 4.639.399.000

## 2. Ma trận cohort (heatmap)

In [ ]:
# ---------- 2. Cohort ----------
first = clean.groupby("customer_id").agg(cohort_idx=("month_idx", "min"), cohort=("ym", "min")).reset_index()
first = first.merge(customers[["customer_id", "acquisition_channel"]], on="customer_id")
act = clean[["customer_id", "month_idx"]].drop_duplicates().merge(first, on="customer_id")
act["m"] = act["month_idx"] - act["cohort_idx"]
LAST = 2025 * 12 + 12

size = act[act["m"] == 0].groupby("cohort")["customer_id"].nunique()
mat = act.pivot_table(index="cohort", columns="m", values="customer_id", aggfunc="nunique")
ret = mat.div(size, axis=0) * 100
cidx = first.groupby("cohort")["cohort_idx"].first()
for k in ret.columns:                                     # ô chưa đủ thời gian theo dõi -> NaN
    ret.loc[cidx[cidx + k > LAST].index, k] = np.nan
ret = ret.loc[:, 1:12]

fig, ax = plt.subplots(figsize=(11, 7.5))
im = ax.imshow(ret.values, cmap="Blues", vmin=0, vmax=30, aspect="auto")
for i in range(ret.shape[0]):
    for j in range(ret.shape[1]):
        v = ret.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 20 else "#1B2B44")
ax.set_xticks(range(12), [f"M{k}" for k in range(1, 13)])
ax.set_yticks(range(len(ret)), [f"{c}  (n={size[c]})" for c in ret.index], fontsize=8)
ax.set_xlabel("Số tháng sau lần mua đầu tiên")
ax.set_title("Tỷ lệ khách quay lại mua (%) theo cohort tháng mua đầu tiên", fontsize=13, loc="left")
r06 = list(ret.index).index("2025-06")                   # cohort 06/2025: M1 rơi vào 07/2025
ax.axhline(r06 - 0.5, color=ORANGE, lw=1.2, ls="--")
ax.text(7.6, r06 - 1, "↓ Từ cohort 06/2025: M1 rơi vào\nsau khi tăng phí ship (07/2025)",
        color="#A8560F", fontsize=8.5, ha="left", va="center")
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.6, label="% khách quay lại")
plt.tight_layout(); plt.savefig(IMG / "cohort_heatmap.png", dpi=150); plt.show()

## 3. Giữ chân theo kênh thu hút

In [ ]:
# ---------- 3. Đường giữ chân theo kênh (cohort 01/2024 - 06/2025, đủ 6 tháng theo dõi) ----------
sub = act[act["cohort_idx"] <= 2025 * 12 + 6]
base = sub[sub["m"] == 0].groupby("acquisition_channel")["customer_id"].nunique()
curve = (sub[sub["m"].between(0, 6)].groupby(["acquisition_channel", "m"])["customer_id"]
         .nunique().unstack().div(base, axis=0) * 100)
order_ch = curve[6].sort_values(ascending=False).index
colors = {"Giới thiệu": "#1F5A93", "Tự nhiên": "#5B9BD5", "Google Search": "#9AA5B1",
          "Facebook Ads": "#E8A15C", "TikTok Shop": ORANGE}
fig, ax = plt.subplots(figsize=(9, 5))
for ch in order_ch:
    y = curve.loc[ch, 1:6]
    ax.plot(y.index, y.values, marker="o", lw=2, color=colors[ch])
    ax.text(6.12, y.iloc[-1], f"{ch} ({y.iloc[-1]:.0f}%)", va="center", fontsize=9, color="#1B2B44")
ax.set_xticks(range(1, 7), [f"M{k}" for k in range(1, 7)])
ax.set_xlim(0.8, 7.6)
ax.set_ylabel("% khách mua trong tháng")
ax.set_title("Khách từ Giới thiệu quay lại gấp ~3 lần khách từ TikTok Shop", fontsize=13, loc="left")
plt.tight_layout(); plt.savefig(IMG / "retention_by_channel.png", dpi=150); plt.show()

## 4. RFM
Bảng in ra phải khớp mục 4.3 của `sql/04_rfm.sql`.

In [ ]:
# ---------- 4. RFM: đúng quy tắc của sql/04_rfm.sql ----------
def ntile(order_series, n=5):
    """Giống NTILE(n) của SQL: chia theo thứ tự, các nhóm đầu nhận dư."""
    N = len(order_series); q, r = divmod(N, n)
    sizes = [q + 1 if i < r else q for i in range(n)]
    return np.repeat(np.arange(1, n + 1), sizes)

rfm = clean.groupby("customer_id").agg(last=("order_date", "max"), frequency=("order_id", "count"),
                                       monetary=("net_value", "sum")).reset_index()
rfm["recency_days"] = (pd.Timestamp("2026-01-01") - rfm["last"].dt.normalize()).dt.days
r = rfm.sort_values(["recency_days", "customer_id"], ascending=[False, True]); rfm.loc[r.index, "r_score"] = ntile(r)
m = rfm.sort_values(["monetary", "customer_id"]);                               rfm.loc[m.index, "m_score"] = ntile(m)
rfm["f_score"] = pd.cut(rfm["frequency"], [0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)

def segment(r, f):
    if r >= 4 and f >= 4: return "1 Champions"
    if (r >= 4 and f == 3) or (r == 3 and f >= 3): return "2 Trung thành"
    if r >= 4: return "3 Mới / Tiềm năng"
    if r == 3: return "4 Cần chú ý"
    if f >= 4: return "5 Không thể mất"
    if f == 3: return "6 Có nguy cơ rời bỏ"
    return "7 Đã rời bỏ"
rfm["segment"] = [segment(a, b) for a, b in zip(rfm["r_score"], rfm["f_score"])]

seg = rfm.groupby("segment").agg(khach=("customer_id", "count"), dt=("monetary", "sum"))
seg = (seg / seg.sum() * 100).sort_index(ascending=False)
print(seg.round(1))                                         # phải khớp mục 4.3 của SQL

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(seg)); h = 0.38
ax.barh(y + h/2, seg["khach"], h, color=GREY, label="% số khách")
ax.barh(y - h/2, seg["dt"], h, color=BLUE, label="% doanh thu")
for i, (k, d) in enumerate(zip(seg["khach"], seg["dt"])):
    ax.text(k + 0.4, i + h/2, f"{k:.1f}%", va="center", fontsize=8, color="#4A5568")
    ax.text(d + 0.4, i - h/2, f"{d:.1f}%", va="center", fontsize=8, color="#1B2B44")
ax.set_yticks(y, [s[2:] for s in seg.index])
ax.set_xlabel("%")
ax.legend(frameon=False, loc="center right")
ax.set_title("8% khách Champions tạo 28% doanh thu", fontsize=13, loc="left")
plt.tight_layout(); plt.savefig(IMG / "rfm_segments.png", dpi=150); plt.show()

## 5. Tỷ lệ quay lại M1: cùng tháng cohort, 2024 so với 2025

In [ ]:
# ---------- 5. M1 cùng tháng cohort: 2024 vs 2025 ----------
m1 = (act[act["m"].isin([0, 1])].groupby(["cohort", "m"])["customer_id"].nunique().unstack())
m1 = (m1[1] / m1[0] * 100).rename("m1").reset_index()
m1["nam"], m1["thang"] = m1["cohort"].str[:4], m1["cohort"].str[5:].astype(int)
p = m1[m1["thang"] <= 11].pivot(index="thang", columns="nam", values="m1")
fig, ax = plt.subplots(figsize=(9, 4.8))
x = np.arange(1, 12); w = 0.38
ax.bar(x - w/2, p["2024"], w, color=GREY, label="Cohort năm 2024")
ax.bar(x + w/2, p["2025"], w, color=BLUE, label="Cohort năm 2025")
ax.axvspan(5.5, 11.5, color=ORANGE, alpha=0.08)
ax.text(8.5, 25.5, "M1 rơi vào sau 07/2025 (tăng phí ship)", ha="center", color="#A8560F", fontsize=9)
ax.set_xticks(x, [f"T{t}" for t in x])
ax.set_xlabel("Tháng của cohort"); ax.set_ylabel("% quay lại tháng kế tiếp (M1)")
ax.set_ylim(0, 28)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Từ 07/2025, khách mới quay lại ít hơn hẳn cùng kỳ năm trước", fontsize=13, loc="left")
plt.tight_layout(); plt.savefig(IMG / "m1_yoy.png", dpi=150); plt.show()